In [ ]:
# -*- coding: utf-8 -*-
# Allrecipes → Single-row CSV with Nutrition amounts only (JSON-LD first, UI fallback)

import requests, json, re, urllib3, csv
from bs4 import BeautifulSoup
from typing import Any, Dict, List, Optional, Tuple

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

URL = "https://www.allrecipes.com/recipe/128750/chinese-broccoli/"
UA = {"User-Agent": "recipe-extractor/values-only/1.1 (+contact@example.com)"}

NUTRIENT_ORDER: List[str] = [
    "calories","total_fat","saturated_fat","cholesterol","sodium",
    "total_carbohydrate","dietary_fiber","total_sugars","protein",
    "vitamin_c","calcium","iron","potassium",
]

LABELS: Dict[str, List[str]] = {
    "calories": ["Calories"],
    "total_fat": ["Total Fat","Fat"],
    "saturated_fat": ["Saturated Fat","Sat Fat"],
    "cholesterol": ["Cholesterol"],
    "sodium": ["Sodium"],
    "total_carbohydrate": ["Total Carbohydrate","Total Carbs","Carbohydrates","Carbs"],
    "dietary_fiber": ["Dietary Fiber","Fiber"],
    "total_sugars": ["Total Sugars","Sugars"],
    "protein": ["Protein"],
    "vitamin_c": ["Vitamin C","Vit C"],
    "calcium": ["Calcium"],
    "iron": ["Iron"],
    "potassium": ["Potassium"],
}

JSONLD_TO_CANON = {
    "calories": "calories",
    "fatContent": "total_fat",
    "saturatedFatContent": "saturated_fat",
    "cholesterolContent": "cholesterol",
    "sodiumContent": "sodium",
    "carbohydrateContent": "total_carbohydrate",
    "fiberContent": "dietary_fiber",
    "sugarContent": "total_sugars",
    "proteinContent": "protein",
}

# ---------- helpers ----------
def _iso8601_to_minutes(iso: Optional[str]) -> Optional[int]:
    if not iso or not isinstance(iso, str): return None
    m = re.search(r"P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?)?", iso.strip(), re.I)
    if not m: return None
    d = int(m.group(1) or 0); h = int(m.group(2) or 0); mi = int(m.group(3) or 0)
    return d*24*60 + h*60 + mi

def _minutes_to_hhmm(m: Optional[int]) -> Optional[str]:
    if m is None: return None
    h, mm = divmod(m, 60)
    return f"{h}h {mm}m" if h else f"{mm}m"

def _parse_servings(yield_field: Any) -> Optional[str]:
    if not yield_field: return None
    if isinstance(yield_field, list) and yield_field: yield_field = yield_field[0]
    s = re.sub(r"\s+", " ", str(yield_field).strip())
    return s or None

def _to_text_list(x: Any) -> List[str]:
    out: List[str] = []
    if not x: return out
    if isinstance(x, list):
        for it in x:
            if isinstance(it, dict): out.append(it.get("text") or it.get("name") or str(it))
            else: out.append(str(it))
    elif isinstance(x, dict):
        out.append(x.get("text") or x.get("name") or str(x))
    else:
        out.append(str(x))
    return [i.strip() for i in out if str(i).strip()]

def _flatten_instructions(instr: Any) -> List[str]:
    steps: List[str] = []
    if not instr: return steps
    def _t(v: Any) -> Optional[str]:
        if isinstance(v, dict): return v.get("text") or v.get("name")
        return str(v)
    if isinstance(instr, list):
        for item in instr:
            if isinstance(item, dict) and item.get("@type") in ("HowToSection", ["HowToSection"]):
                sub = item.get("itemListElement") or item.get("steps"); steps += _to_text_list(sub)
            else:
                tt = _t(item)
                if tt: steps.append(tt)
    elif isinstance(instr, dict):
        if instr.get("@type") in ("HowToSection", ["HowToSection"]):
            sub = instr.get("itemListElement") or instr.get("steps"); steps += _to_text_list(sub)
        else:
            tt = _t(instr)
            if tt: steps.append(tt)
    else:
        steps += _to_text_list(instr)
    return [s for s in (s.strip() for s in steps) if s]

# ---------- JSON-LD ----------
def extract_recipe_jsonld(html: str) -> Optional[Dict[str, Any]]:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.find_all("script", type="application/ld+json"):
        if not tag.string: continue
        try:
            data = json.loads(tag.string)
        except Exception:
            continue
        for node in _iter_json_objects(data):
            if isinstance(node, dict):
                t = node.get("@type")
                if t == "Recipe" or (isinstance(t, list) and "Recipe" in t):
                    return node
                me = node.get("mainEntity")
                if isinstance(me, dict):
                    mt = me.get("@type")
                    if mt == "Recipe" or (isinstance(mt, list) and "Recipe" in mt):
                        return me
    return None

def _iter_json_objects(obj: Any):
    yield obj
    if isinstance(obj, dict):
        for v in obj.values():
            if isinstance(v, (dict, list)): yield from _iter_json_objects(v)
    elif isinstance(obj, list):
        for e in obj:
            if isinstance(e, (dict, list)): yield from _iter_json_objects(e)

def _normalize_amount_str(s: Optional[str], *, is_calories: bool=False) -> Optional[str]:
    if not s: return None
    if is_calories:
        m = re.search(r"([0-9]+(?:\.[0-9]+)?)", s)
        return f"{m.group(1)}kcal" if m else None
    m = re.search(r"([0-9]+(?:\.[0-9]+)?)\s*([a-zA-Zµ]+)", s)
    if not m: return None
    val = m.group(1); unit = m.group(2).lower()
    if unit == "cal": unit = "kcal"
    return f"{val}{unit}"

def normalize_nutrition_jsonld(n: Any) -> Dict[str, str]:
    out: Dict[str, str] = {}
    if isinstance(n, dict):
        for k, v in n.items():
            canon = JSONLD_TO_CANON.get(k)
            if canon and v is not None:
                if canon == "calories":
                    out[canon] = _normalize_amount_str(str(v), is_calories=True) or str(v)
                else:
                    out[canon] = _normalize_amount_str(str(v)) or str(v)
    return out

# ---------- UI fallback (tight window right after label) ----------
def parse_nutrition_ui_amounts(html: str) -> Dict[str, Optional[str]]:
    soup = BeautifulSoup(html, "html.parser")
    txt = soup.get_text(" ", strip=True)
    m = re.search(r"\bNutrition Facts\b", txt, re.I)
    if not m: return {}
    start = m.start()
    # Trim to a chunk after the header; normalize spaces (incl. NBSP)
    section = txt[start:start+4000]
    section = re.sub(r"[\s\u00A0]+", " ", section).strip()

    results: Dict[str, Optional[str]] = {c: None for c in NUTRIENT_ORDER}

    for canon in NUTRIENT_ORDER:
        pos = None
        for label in LABELS[canon]:
            pat = r"\b" + r"[\s\u00A0]+".join(map(re.escape, label.split())) + r"\b"
            mm = re.search(pat, section, re.I)
            if mm:
                pos = mm.end()
                break
        if pos is None:
            continue

        # Search for amount in a short window after the label (reduces false hits)
        window = section[pos:pos+120]

        if canon == "calories":
            m_amt = re.search(r"\b([0-9]{1,5})\b", window)
            if m_amt:
                results[canon] = f"{m_amt.group(1)}kcal"
            continue

        m_amt = re.search(r"\b([0-9]+(?:\.[0-9]+)?)\s*(g|mg|mcg|µg|kcal|cal|kj)\b", window, re.I)
        if m_amt:
            val = m_amt.group(1)
            unit = m_amt.group(2).lower()
            if unit == "cal": unit = "kcal"
            results[canon] = f"{val}{unit}"

    return results

# ---------- main ----------
def extract_recipe(url: str) -> Dict[str, Any]:
    resp = requests.get(url, headers=UA, timeout=30, verify=False)
    resp.raise_for_status()

    j = extract_recipe_jsonld(resp.text)
    if not j: raise RuntimeError("Recipe JSON-LD not found.")

    title = j.get("name")
    servings = _parse_servings(j.get("recipeYield"))
    prep = _minutes_to_hhmm(_iso8601_to_minutes(j.get("prepTime")))
    cook = _minutes_to_hhmm(_iso8601_to_minutes(j.get("cookTime")))
    total = _minutes_to_hhmm(_iso8601_to_minutes(j.get("totalTime")))

    ingredients = _to_text_list(j.get("recipeIngredient"))
    directions  = _flatten_instructions(j.get("recipeInstructions"))

    # NEW: JSON-LD FIRST, UI as fallback
    jsonld_amounts = normalize_nutrition_jsonld(j.get("nutrition", {}))
    ui_amounts     = parse_nutrition_ui_amounts(resp.text)

    merged: Dict[str, Optional[str]] = {}
    for canon in NUTRIENT_ORDER:
        merged[canon] = jsonld_amounts.get(canon) or ui_amounts.get(canon) or None

    return {
        "url": url,
        "title": title,
        "prepTime": prep,
        "cookTime": cook,
        "totalTime": total,
        "servings": servings,
        "ingredients": ingredients,
        "directions": directions,
        "nutrition_amounts": merged,
    }

# ---------- CSV ----------
def to_one_row(data: Dict[str, Any], list_sep: str = " | ") -> Dict[str, Any]:
    def nv(v):
        if v is None or (isinstance(v, str) and not v.strip()):
            return "not provided"
        if isinstance(v, (list, tuple)) and not v:
            return "not provided"
        return v

    row = {
        "title": nv(data.get("title")),
        "url": nv(data.get("url")),
        "servings": nv(data.get("servings")),
        "prepTime": nv(data.get("prepTime")),
        "cookTime": nv(data.get("cookTime")),
        "totalTime": nv(data.get("totalTime")),
        "ingredients": nv(" | ".join(data.get("ingredients") or [])),
        "directions": nv(" | ".join(f"{i+1}. {s}" for i, s in enumerate(data.get("directions") or [], 1))),
    }
    amounts = data.get("nutrition_amounts", {}) or {}
    for canon in NUTRIENT_ORDER:
        row[f"nutrition_{canon}"] = nv(amounts.get(canon))
    return row

def write_one_row_csv(path: str, row: Dict[str, Any], order: Optional[List[str]] = None) -> None:
    if order is None:
        order = list(row.keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=order)
        w.writeheader(); w.writerow(row)

if __name__ == "__main__":
    data = extract_recipe(URL)

    print("TITLE:", data["title"])
    print("Servings:", data["servings"])
    print("Times:", data["prepTime"], "/", data["cookTime"], "/", data["totalTime"])
    print("\nNUTRITION (amounts):")
    for k in NUTRIENT_ORDER:
        print(f"- {k}: {data['nutrition_amounts'].get(k) or 'not provided'}")

    order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
            [f"nutrition_{c}" for c in NUTRIENT_ORDER]
    write_one_row_csv("recipe_single_row.csv", to_one_row(data), order)
    print("\nWrote recipe_single_row.csv with 1 row.")


: 

In [ ]:
resp = requests.get("https://www.allrecipes.com/", headers=UA, timeout=30, verify=False)
soup = BeautifulSoup(resp.text, "html.parser")
recipe_links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]
    if href.startswith("https://www.allrecipes.com/recipe/"):
        recipe_links.add(href.split("?")[0])  # Remove query params

print(f"Found {len(recipe_links)} recipe URLs.")
for url in list(recipe_links)[:100]:  # Show first 10 as example
    print(url)

In [ ]:
data = extract_recipe("https://www.allrecipes.com/recipe/128750/chinese-broccoli/")

In [ ]:
print(data)

In [ ]:
order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
            [f"nutrition_{c}" for c in NUTRIENT_ORDER]
write_one_row_csv("recipe_single_row.csv", to_one_row(data), order)

In [ ]:
all_data = []
for url in recipe_links:
    try:
        data = extract_recipe(url)
        all_data.append(data)
    except Exception as e:
        print(f"Failed to extract {url}: {e}")
print(f"Extracted data from {len(all_data)} recipes.")

In [ ]:
order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
    [f"nutrition_{c}" for c in NUTRIENT_ORDER]

with open("all_recipes_single_row.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=order)
    writer.writeheader()
    for d in all_data:
        writer.writerow(to_one_row(d))

In [ ]:
resp = requests.get("https://www.allrecipes.com/recipes/695/world-cuisine/asian/chinese/", headers=UA, timeout=30, verify=False)
soup = BeautifulSoup(resp.text, "html.parser")
links = set()
for a in soup.find_all("a", href=True):
    href = a["href"]
    if href.startswith("https://www.allrecipes.com/recipe/"):
        links.add(href.split("?")[0])
print(f"Found {len(links)} recipe links.")
for link in list(links)[:100]:
    print(link)

In [ ]:
all_data = []
for url in links:
    try:
        data = extract_recipe(url)
        all_data.append(data)
    except Exception as e:
        print(f"Failed to extract {url}: {e}")
print(f"Extracted data from {len(all_data)} recipes.")

In [ ]:
print(all_data[0])

In [ ]:
order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
    [f"nutrition_{c}" for c in NUTRIENT_ORDER]

with open("all_recipes_Chinese.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=order)
    writer.writeheader()
    for d in all_data:
        writer.writerow(to_one_row(d))

In [ ]:
resp = requests.get(URL, headers={"User-Agent": 'recipe-extractor/values-only/1.1 (+contact@example.com)'}, timeout=30, verify=False)
soup = BeautifulSoup(resp.text, "html.parser")
cuisine_links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]
    if href.startswith("https://www.allrecipes.com/recipes/") and "/world-cuisine/" in href:
        cuisine_links.add(href.split("?")[0])

print(f"Found {len(cuisine_links)} cuisine links.")
for link in sorted(cuisine_links):
    print(link)

In [ ]:
cuisine_links

In [ ]:
with open("cuisine_links.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["url"])
    for link in cuisine_links:
        writer.writerow([link])

In [ ]:
all_recipe_links = set()
for cuisine_url in cuisine_links:
    try:
        resp = requests.get(cuisine_url, headers=UA, timeout=30, verify=False)
        soup = BeautifulSoup(resp.text, "html.parser")
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if href.startswith("https://www.allrecipes.com/recipe/"):
                all_recipe_links.add(href.split("?")[0])
    except Exception as e:
        print(f"Failed to process {cuisine_url}: {e}")

print(f"Found {len(all_recipe_links)} total recipe links from all cuisines.")
for link in list(all_recipe_links)[:10000]:
    print(link)



In [ ]:
all_recipe_links
print(len(all_recipe_links))

In [ ]:
import csv

with open("all_recipe_links.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["url"])
    for link in all_recipe_links:
        writer.writerow([link])

In [ ]:
all_data = []
for url in all_recipe_links:
    try:
        data = extract_recipe(url)
        all_data.append(data)
    except Exception as e:
        print(f"Failed to extract {url}: {e}")
print(f"Extracted data from {len(all_data)} recipes.")

In [ ]:
print(all_data[0])

In [ ]:
order = ["title","url","servings","prepTime","cookTime","totalTime","ingredients","directions"] + \
    [f"nutrition_{c}" for c in NUTRIENT_ORDER]

with open("all_recipes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=order)
    writer.writeheader()
    for d in all_data:
        writer.writerow(to_one_row(d))